In [82]:
import numpy as np 
import pandas as pd

In [83]:
transactions = pd.read_csv("dataset/project_transactions.csv")
transactions.head()

,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,1364,26984896261,1,842930,1,2.19,31742,0.00,1,0.0,0.0
1,1364,26984896261,1,897044,1,2.99,31742,-0.40,1,0.0,0.0
2,1364,26984896261,1,920955,1,3.09,31742,0.00,1,0.0,0.0
3,1364,26984896261,1,937406,1,2.50,31742,-0.99,1,0.0,0.0
4,1364,26984896261,1,981760,1,0.60,31742,-0.79,1,0.0,0.0


## Data Exploration

### Data Reduction

In [84]:
# Transactions max "DAY" column value is 711 so DAY column can be reduced from int64 to int16
# Transactions max "QUANTITY" column value is 89638 so QUANTITY column can be reduced from int64 to int32
# Transactions max "STORE_ID" column value is 34280 so STORE_ID column can be reduced from int64 to int32
# Transactions max "WEEK_NO" column value is 102 so WEEK_NO column can be reduced from int64 to int8
print(transactions.loc[:, "DAY"].max())
print(transactions.loc[:, "QUANTITY"].max())
print(transactions.loc[:, "STORE_ID"].max())
print(transactions.loc[:, "WEEK_NO"].max())

711
89638
34280
102


In [85]:
transactions.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2146311 entries, 0 to 2146310
Data columns (total 11 columns):
 #   Column             Dtype  
---  ------             -----  
 0   household_key      int64  
 1   BASKET_ID          int64  
 2   DAY                int64  
 3   PRODUCT_ID         int64  
 4   QUANTITY           int64  
 5   SALES_VALUE        float64
 6   STORE_ID           int64  
 7   RETAIL_DISC        float64
 8   WEEK_NO            int64  
 9   COUPON_DISC        float64
 10  COUPON_MATCH_DISC  float64
dtypes: float64(4), int64(7)
memory usage: 180.1 MB


In [86]:
transactions = transactions.astype({
                        "DAY":"int16",
                        "QUANTITY":"int32",
                        "STORE_ID":"int32",
                        "WEEK_NO":"int8"
                    })

In [87]:
# drop from 180.1mb to 137.1mb
transactions.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2146311 entries, 0 to 2146310
Data columns (total 11 columns):
 #   Column             Dtype  
---  ------             -----  
 0   household_key      int64  
 1   BASKET_ID          int64  
 2   DAY                int16  
 3   PRODUCT_ID         int64  
 4   QUANTITY           int32  
 5   SALES_VALUE        float64
 6   STORE_ID           int32  
 7   RETAIL_DISC        float64
 8   WEEK_NO            int8   
 9   COUPON_DISC        float64
 10  COUPON_MATCH_DISC  float64
dtypes: float64(4), int16(1), int32(2), int64(3), int8(1)
memory usage: 137.1 MB


### Missing data

In [88]:
# No missing data
transactions.isna().sum()

household_key        0
BASKET_ID            0
DAY                  0
PRODUCT_ID           0
QUANTITY             0
SALES_VALUE          0
STORE_ID             0
RETAIL_DISC          0
WEEK_NO              0
COUPON_DISC          0
COUPON_MATCH_DISC    0
dtype: int64

### Unique households and products

In [89]:
# 2099 unique households
# 84138 unique products
transactions.loc[:, ["household_key", "PRODUCT_ID"]].nunique()

household_key     2099
PRODUCT_ID       84138
dtype: int64

## Column Creation

In [90]:
transactions["total_discount"] = transactions["RETAIL_DISC"] + transactions["COUPON_DISC"]

conditions = [
    ((transactions["total_discount"] / transactions["SALES_VALUE"]) > 1),
    ((transactions["total_discount"] / transactions["SALES_VALUE"]) < 0)
]
choices = [1, 0]

transactions["discount_pct"] = np.select(conditions, choices, (transactions["total_discount"] / transactions["SALES_VALUE"]))

In [91]:
transactions = transactions.drop(columns=["RETAIL_DISC", "COUPON_DISC", "COUPON_MATCH_DISC"])
transactions

,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,WEEK_NO,total_discount,discount_pct
0,1364,26984896261,1,842930,1,2.19,31742,1,0.00,0.0
1,1364,26984896261,1,897044,1,2.99,31742,1,-0.40,0.0
2,1364,26984896261,1,920955,1,3.09,31742,1,0.00,0.0
3,1364,26984896261,1,937406,1,2.50,31742,1,-0.99,0.0
4,1364,26984896261,1,981760,1,0.60,31742,1,-0.79,0.0
...,...,...,...,...,...,...,...,...,...,...
2146306,1598,42305362535,711,92130,1,0.99,3228,102,0.00,0.0
2146307,1598,42305362535,711,114102,1,8.89,3228,102,0.00,0.0
2146308,1598,42305362535,711,133449,1,6.99,3228,102,0.00,0.0
2146309,1598,42305362535,711,6923644,1,4.50,3228,102,-0.49,0.0


## Overall Statistics